# Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

RANDOM_STATE = 42

In [2]:
df = pd.read_csv(r"C:\Users\ASUS\OneDrive\Desktop\Datase\crop_yield.csv")

print("Fertilizer raw stats:\n", df['Fertilizer'].describe())
print("\nFertilizer / Area (per-hectare-equivalent):\n", (df['Fertilizer'] / df['Area']).describe())
print("\nPesticide raw stats:\n", df['Pesticide'].describe())
print("\nPesticide / Area:\n", (df['Pesticide'] / df['Area']).describe())

Fertilizer raw stats:
 count    1.968900e+04
mean     2.410331e+07
std      9.494600e+07
min      5.417000e+01
25%      1.880146e+05
50%      1.234957e+06
75%      1.000385e+07
max      4.835407e+09
Name: Fertilizer, dtype: float64

Fertilizer / Area (per-hectare-equivalent):
 count    19689.000000
mean       137.160208
std         26.200647
min         94.670000
25%        108.340000
50%        144.490000
75%        157.910000
max        193.610000
dtype: float64

Pesticide raw stats:
 count    1.968900e+04
mean     4.884835e+04
std      2.132874e+05
min      9.000000e-02
25%      3.567000e+02
50%      2.421900e+03
75%      2.004170e+04
max      1.575051e+07
Name: Pesticide, dtype: float64

Pesticide / Area:
 count    19689.000000
mean         0.274374
std          0.073284
min          0.090000
25%          0.220000
50%          0.270000
75%          0.330000
max          0.380000
dtype: float64


In [3]:
df['Fertilizer_per_area'] = df['Fertilizer'] / df['Area']
df['Pesticide_per_area'] = df['Pesticide'] / df['Area']

feature_cols = ['Crop', 'State', 'Season', 'Crop_Year', 'Annual_Rainfall',
                 'Fertilizer_per_area', 'Pesticide_per_area', 'Area']
target_col = 'Yield'

df_clean = df.dropna(subset=feature_cols + [target_col]).copy()

# Now clip outliers on the NORMALIZED columns, not the raw totals
for col in ['Yield', 'Fertilizer_per_area', 'Pesticide_per_area', 'Annual_Rainfall']:
    q1, q99 = df_clean[col].quantile([0.01, 0.99])
    df_clean = df_clean[(df_clean[col] >= q1) & (df_clean[col] <= q99)]

print("Rows after cleaning:", df_clean.shape[0], "/ original:", df.shape[0])
print(df_clean[['Fertilizer_per_area', 'Pesticide_per_area']].describe())

Rows after cleaning: 18782 / original: 19689
       Fertilizer_per_area  Pesticide_per_area
count         18782.000000        18782.000000
mean            137.433459            0.274001
std              25.927162            0.073716
min              94.670000            0.090000
25%             108.340000            0.220000
50%             144.490000            0.270000
75%             157.910000            0.330000
max             171.760000            0.380000


In [4]:
df_clean['Yield_log'] = np.log1p(df_clean['Yield'])

le_crop = LabelEncoder()
le_state = LabelEncoder()
le_season = LabelEncoder()

df_clean['Crop_enc'] = le_crop.fit_transform(df_clean['Crop'])
df_clean['State_enc'] = le_state.fit_transform(df_clean['State'])
df_clean['Season_enc'] = le_season.fit_transform(df_clean['Season'].str.strip())

X = df_clean[['Crop_enc', 'State_enc', 'Season_enc', 'Crop_Year',
              'Annual_Rainfall', 'Fertilizer_per_area', 'Pesticide_per_area', 'Area']]
y = df_clean['Yield_log']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (15025, 8) Test: (3757, 8)


In [8]:
model = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
model.fit(X_train, y_train)

y_pred_log = model.predict(X_test)
y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
r2 = r2_score(y_test, y_pred_log)

print(f"MAE (real units): {mae:.4f}")
print(f"RMSE (real units): {rmse:.4f}")
print(f"R² (log scale): {r2:.4f}")

MAE (real units): 0.7925
RMSE (real units): 2.9491
R² (log scale): 0.9336


In [9]:
# Pick the exact row that broke last time and confirm it's now sane
check_row = df_clean[(df_clean['Crop'] == 'Rice') & (df_clean['State'] == 'Uttar Pradesh') &
                     (df_clean['Season'].str.strip() == 'Kharif')].sort_values('Crop_Year').tail(1)

if len(check_row) == 0:
    print("This exact combination was filtered out as an outlier — check df_clean before this row was dropped")
else:
    actual = check_row.iloc[0]
    test_input = pd.DataFrame([{
        "Crop_enc": le_crop.transform(["Rice"])[0],
        "State_enc": le_state.transform(["Uttar Pradesh"])[0],
        "Season_enc": le_season.transform(["Kharif"])[0],
        "Crop_Year": actual['Crop_Year'],
        "Annual_Rainfall": actual['Annual_Rainfall'],
        "Fertilizer_per_area": actual['Fertilizer_per_area'],
        "Pesticide_per_area": actual['Pesticide_per_area'],
        "Area": actual['Area']
    }])
    pred = np.expm1(model.predict(test_input)[0])
    print(f"Real yield: {actual['Yield']:.4f} | Model prediction on its own training row: {pred:.4f}")
    print("Sane (within reasonable error)?", abs(pred - actual['Yield']) < actual['Yield'] * 0.5)

Real yield: 2.8235 | Model prediction on its own training row: 2.7111
Sane (within reasonable error)? True


In [10]:
joblib.dump(model, "yield_prediction_RandomForestRegressor_v2.joblib")
joblib.dump(le_crop, "yield_le_crop_v2.joblib")
joblib.dump(le_state, "yield_le_state_v2.joblib")
joblib.dump(le_season, "yield_le_season_v2.joblib")
print("Saved matched artifact set (v2) — model and all three encoders from the SAME training run.")

Saved matched artifact set (v2) — model and all three encoders from the SAME training run.
